In [102]:
# testing hysteresis parameter calculation? 
import numpy as np
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator


storm_directory = 'data/constituents'
storms = {}
for filename in os.listdir(storm_directory):
    # check if the file is a CSV file
    if filename.endswith('.csv'):
        file_path = os.path.join(storm_directory, filename) # construct the full file path
        df = pd.read_csv(file_path)                         # read the CSV file into a data frame
        df = df.dropna(subset=['Date_Time'])                # drop rows where 'Date/Time' is NaN  
        df['Date_Time'] = pd.to_datetime(df['Date_Time'])   # convert to datetime format
        df = df.set_index('Date_Time')                      # set date time as the index 
        df = df.dropna(how='all', axis=1)                   # drop columns where all values are NaN
        key = filename[:-4]                                 # remove the '.csv' from the filename to use as the dictionary key
        storms[key] = df                                    # store the data frame in the dictionary

shear_stress = pd.read_csv('data/shear_stress/average_shear_stress_lajara_full_smoothed.csv', parse_dates=['datetime'], index_col='datetime')
sonde_downstream = pd.read_csv('data/sonde_data/sonde_down_full_record_smoothed.csv', parse_dates=['DateTime'], index_col='DateTime')
sonde_upstream = pd.read_csv('data/sonde_data/sonde_up_full_record_smoothed.csv', parse_dates=['DateTime'], index_col='DateTime')

storms['st5_down']


,SS (uL/L),SSC (mg/L),PP (mg/L),POC (mg/L),DOC (mg/L),LAB ID,Unnamed: 14
Date_Time,,,,,,,
2023-08-13 18:30:00,0.000,13.330000,0.0000,0.090,8.085,NaN,NaN
2023-08-13 18:45:00,210.458,76.153846,0.0683,7.778,4.473,456.0,NaN
2023-08-13 18:58:00,116.066,41.200000,0.0961,4.535,3.936,457.0,NaN
2023-08-13 19:02:00,113.258,39.600000,0.0968,5.889,3.943,458.0,NaN
2023-08-13 19:10:00,81.150,27.600000,0.0906,4.804,4.039,459.0,`
2023-08-13 21:02:00,56.150,20.833333,0.0392,2.921,2.786,460.0,NaN


Process and Merge Data

In [103]:
# average rows with duplicate timestamps on sonde data (second-resolution data)
sonde_downstream = sonde_downstream.groupby(level=0).mean(numeric_only=True)
sonde_upstream = sonde_upstream.groupby(level=0).mean(numeric_only=True)

# re-sample shear stress and sonde data to 1 min intervals
shear_stress = shear_stress.resample('1min').interpolate()
sonde_downstream_resampled = sonde_downstream.resample('1min').interpolate()
sonde_upstream_resampled = sonde_upstream.resample('1min').interpolate()

In [104]:
# join shear stress data and the matching sonde data with the storm data
merged_storms = {}

for storm_name, storm_df in storms.items():
    if "down" in storm_name.lower():
        sonde_df = sonde_downstream_resampled[["Turbidity FNU", "fDOM RFU"]]
    elif "up" in storm_name.lower():
        sonde_df = sonde_upstream_resampled[["Turbidity FNU", "fDOM RFU"]]
    else:
        continue

    merged_storm_df = storm_df.join(shear_stress, how="left")
    merged_storm_df = merged_storm_df.join(sonde_df, how="left")
    merged_storms[storm_name] = merged_storm_df

# also join the sonde data with the shear stress data for the entire record 
sonde_downstream = shear_stress.join(sonde_downstream[["Turbidity FNU", "fDOM RFU"]], how="left")
sonde_upstream = shear_stress.join(sonde_upstream[["Turbidity FNU", "fDOM RFU"]], how="left")

In [105]:
merged_storms['st5_down']

,SS (uL/L),SSC (mg/L),PP (mg/L),POC (mg/L),DOC (mg/L),LAB ID,Unnamed: 14,shear_stress,Turbidity FNU,fDOM RFU
Date_Time,,,,,,,,,,
2023-08-13 18:30:00,0.000,13.330000,0.0000,0.090,8.085,NaN,NaN,63.940969,10.868000,6.4620
2023-08-13 18:45:00,210.458,76.153846,0.0683,7.778,4.473,456.0,NaN,64.934699,13.716000,7.4420
2023-08-13 18:58:00,116.066,41.200000,0.0961,4.535,3.936,457.0,NaN,65.779039,16.154400,9.5616
2023-08-13 19:02:00,113.258,39.600000,0.0968,5.889,3.943,458.0,NaN,65.962161,14.494400,9.6416
2023-08-13 19:10:00,81.150,27.600000,0.0906,4.804,4.039,459.0,`,66.175056,11.944000,9.6900
2023-08-13 21:02:00,56.150,20.833333,0.0392,2.921,2.786,460.0,NaN,67.591869,7.864267,7.7592


Hysteresis index calculation functions

In [106]:
def normalize_data(df, tau_col, constituent_col):
    pair = df[[tau_col, constituent_col]].dropna().sort_index()

    if pair.empty:
        return None

    tau_max = pair[tau_col].max()
    cn_max = pair[constituent_col].max()

    if pd.isna(tau_max) or pd.isna(cn_max) or tau_max == 0 or cn_max == 0:
        return None

    pair = pair.copy()
    pair["tau_n"] = pair[tau_col] / tau_max
    pair["cn"] = pair[constituent_col] / cn_max
    return pair


def split_hydrograph(norm_df):
    if norm_df.empty:
        return None, None, None

    peak_pos = norm_df["tau_n"].to_numpy().argmax()
    peak_label = norm_df.index[peak_pos]

    rise = norm_df.iloc[:peak_pos + 1][["tau_n", "cn"]]
    fall = norm_df.iloc[peak_pos + 1:][["tau_n", "cn"]]

    return rise, fall, peak_label


def reference_line(norm_df, peak_label):
    peak_row = norm_df.loc[peak_label]
    last_row = norm_df.iloc[-1]

    x1 = peak_row["tau_n"]
    y1 = peak_row["cn"]
    x2 = last_row["tau_n"]
    y2 = last_row["cn"]

    A = y1 - y2
    B = x2 - x1
    C = x1 * y2 - x2 * y1
    return A, B, C


def distance_from_line(df_xy, A, B, C):
    return (A * df_xy["tau_n"] + B * df_xy["cn"] + C) / np.sqrt(A**2 + B**2)


def projection_to_line(x0, y0, A, B, C):
    d = (A * x0 + B * y0 + C) / (A**2 + B**2)
    x_proj = x0 - A * d
    y_proj = y0 - B * d
    return x_proj, y_proj


def calculate_aich_HI(df, tau_col, constituent_col, storm_name):
    norm_df = normalize_data(df, tau_col, constituent_col)

    if norm_df is None or len(norm_df) < 3:
        print(f"Not enough paired data to calculate AICH HI for storm {storm_name}")
        return None

    rise, fall, peak_label = split_hydrograph(norm_df)

    if rise is None or fall is None or rise.empty or fall.empty:
        print(f"Not enough data points to calculate AICH HI for storm {storm_name}")
        return None

    A, B, C = reference_line(norm_df, peak_label)

    rise_dist = -distance_from_line(rise, A, B, C)
    fall_dist = distance_from_line(fall, A, B, C)

    if rise_dist.empty or fall_dist.empty:
        print(f"Not enough data points to calculate AICH HI for storm {storm_name}")
        return None

    rise_idx = rise_dist.abs().idxmax()
    fall_idx = fall_dist.abs().idxmax()

    Drise = rise_dist.loc[rise_idx]
    Dfall = fall_dist.loc[fall_idx]
    HI = Drise + Dfall

    results = {
        "HI": HI,
        "Drise": Drise,
        "Dfall": Dfall,
        "storm_name": storm_name,
        "peak tau": norm_df.loc[peak_label, "tau_n"],
        "peak constituent": norm_df.loc[peak_label, "cn"],
        "rise": rise,
        "fall": fall,
        "A": A,
        "B": B,
        "C": C,
        "Drise_point": (rise.loc[rise_idx, "tau_n"], rise.loc[rise_idx, "cn"]),
        "Dfall_point": (fall.loc[fall_idx, "tau_n"], fall.loc[fall_idx, "cn"]),
        "norm_df": norm_df,
    }
    return results




def plot_aich_hysteresis(results, df, tau_col, constituent_col,
                        xlabel=r"Normalized Shear Stress ($\tau_n$)",
                        ylabel="Normalized Constituent",
                        out_dir="HI_calculations/aich_plots",
                        save=True,
                        show=False):
    
    norm_df = results["norm_df"]
    rise = results["rise"]
    fall = results["fall"]
    A = results["A"]
    B = results["B"]
    C = results["C"]
    Drise = results["Drise"]
    Dfall = results["Dfall"]
    HI = results["HI"]
    storm_name = results.get("storm_name", "")

    xr, yr = results["Drise_point"]
    xf, yf = results["Dfall_point"]

    xr_proj, yr_proj = projection_to_line(xr, yr, A, B, C)
    xf_proj, yf_proj = projection_to_line(xf, yf, A, B, C)

    xmin = min(norm_df["tau_n"].min(), xr_proj, xf_proj) - 0.05
    xmax = max(norm_df["tau_n"].max(), xr_proj, xf_proj) + 0.05

    xline = np.linspace(xmin, xmax, 200)
    yline = -(A * xline + C) / B

    ts_df = df[[tau_col, constituent_col]].dropna().sort_index()

    fig, axes = plt.subplots(1, 2, figsize=(10, 5), gridspec_kw={"width_ratios": [1.2, 1]})
    # left: time series
    ax = axes[0]
    ax.plot(ts_df.index, ts_df[tau_col], color="tab:blue", linewidth=1.5, label=tau_col)
    ax.set_ylabel(tau_col, color="tab:blue")
    ax.tick_params(axis="y", labelcolor="tab:blue")
    ax.xaxis.set_major_locator(MaxNLocator(8))

    ax2 = ax.twinx()
    ax2.plot(ts_df.index, ts_df[constituent_col], color="tab:red", linewidth=1.5, label=constituent_col)
    ax2.set_ylabel(constituent_col, color="tab:red")
    ax2.tick_params(axis="y", labelcolor="tab:red")

    ax.set_title("Event time series")
    ax.set_xlabel("Time")
    ax.grid(True, alpha=0.3)

    # Right: hysteresis pattern
    ax = axes[1]
    ax.plot(norm_df["tau_n"], norm_df["cn"], color="black", linewidth=2, alpha=0.7, zorder=1)
    ax.scatter(rise["tau_n"], rise["cn"], color="forestgreen", linewidth=2, label="Rising limb", zorder=4)
    ax.scatter(fall["tau_n"], fall["cn"], color="firebrick", linewidth=2, label="Falling limb", zorder=4)
    ax.plot(xline, yline, "--", color="gray", linewidth=2, label="Reference line", zorder=2)

    ax.scatter(xr, yr, s=80, color="forestgreen", zorder=5)
    ax.plot([xr, xr_proj], [yr, yr_proj], "--", color="forestgreen", linewidth=2, zorder=3)
    ax.scatter(xf, yf, s=80, color="firebrick", zorder=5)
    ax.plot([xf, xf_proj], [yf, yf_proj], "--", color="firebrick", linewidth=2, zorder=3)

    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)

    # make axes limits according to the data range, but with some padding
    geom_x = np.r_[norm_df["tau_n"].to_numpy(), xr, xr_proj, xf, xf_proj]
    geom_y = np.r_[norm_df["cn"].to_numpy(), yr, yr_proj, yf, yf_proj]
    x_min, x_max = geom_x.min(), geom_x.max()
    y_min, y_max = geom_y.min(), geom_y.max()
    x_span = x_max - x_min
    y_span = y_max - y_min
    span = max(x_span, y_span)
    x_mid = 0.5 * (x_min + x_max)
    y_mid = 0.5 * (y_min + y_max)
    pad = 0.05 * span
    ax.set_xlim(x_mid - span / 2 - pad, x_mid + span / 2 + pad)
    ax.set_ylim(y_mid - span / 2 - pad, y_mid + span / 2 + pad)
    ax.set_aspect("equal", adjustable="box")

    ax.grid(True, alpha=0.3)
    ax.legend()
    ax.set_title(f"Drise = {Drise:.2f}, Dfall = {Dfall:.2f}, HI = {HI:.2f}")

    main_title = f"{storm_name} - {constituent_col} Hysteresis" if storm_name else f"{constituent_col} Hysteresis"
    fig.suptitle(main_title, fontsize=15)
    fig.tight_layout(rect=[0, 0, 1, 0.95])

    if save:
        os.makedirs(out_dir, exist_ok=True)
        safe_name = f"{storm_name}_{constituent_col}_langlois.png".replace(" ", "_").replace("/", "_")
        fig.savefig(os.path.join(out_dir, safe_name), dpi=300, bbox_inches="tight")

    if show:
        plt.show()

    plt.close(fig)

Calculate constituents hysteresis 

In [107]:
all_results = []

for storm_name, storm_df in merged_storms.items():
    for constituent in ["SSC (mg/L)", "SRP (mg/L)", "TP (mg/L)", "PP (mg/L)", "POC (mg/L)", "DOC (mg/L)"]:
        if constituent not in storm_df.columns:
            continue
        result = calculate_aich_HI(
            storm_df,
            tau_col="shear_stress",
            constituent_col=constituent,
            storm_name=storm_name)

        if result is not None:
            result["storm"] = storm_name
            result["constituent"] = constituent
            all_results.append(result)

all_results = pd.DataFrame(all_results)
all_results.to_csv('HI_calculations/aich_hysteresis_results.csv', index=False)

Not enough data points to calculate AICH HI for storm st5_down
Not enough data points to calculate AICH HI for storm st5_down
Not enough data points to calculate AICH HI for storm st5_down
Not enough data points to calculate AICH HI for storm st5_down


Plot constituents

In [108]:
for storm_name, storm_df in merged_storms.items():
    for constituent in ["SSC (mg/L)", "SRP (mg/L)", "TP (mg/L)", "PP (mg/L)", "POC (mg/L)", "DOC (mg/L)"]:
        if constituent not in storm_df.columns:
            continue

        result = calculate_aich_HI(
            storm_df,
            tau_col="shear_stress",
            constituent_col=constituent,
            storm_name=storm_name
        )

        if result is not None:
            plot_aich_hysteresis(result, storm_df, "shear_stress", constituent)

Not enough data points to calculate AICH HI for storm st5_down
Not enough data points to calculate AICH HI for storm st5_down
Not enough data points to calculate AICH HI for storm st5_down
Not enough data points to calculate AICH HI for storm st5_down


### Turbidity and fDOM hysteresis

Separating storm by date and time

In [109]:
# define storm windows once
storm_windows = {
    "st1_down": ("2021-07-23 12:30", "2021-07-28 11:15"),
    "st1_up":   ("2021-07-23 12:30", "2021-07-28 11:15"),
    "st2_down": ("2022-08-03 15:00", "2022-08-03 19:00"),
    "st2_up":   ("2022-08-03 15:00", "2022-08-03 19:00"),
    "st3_down": ("2022-08-08 12:30", "2022-08-08 22:15"),
    "st3_up":   ("2022-08-08 12:30", "2022-08-08 22:15"),
    "st4_down": ("2023-07-29 13:00", "2023-07-30 10:30"),
    "st4_up":   ("2023-07-29 13:00", "2023-07-30 10:30"),
    "st5_down": ("2023-08-13 17:15", "2023-08-14 03:15"),
    "st5_up":   ("2023-08-13 17:15", "2023-08-14 03:15"),
    "st6_down": ("2023-08-28 11:30", "2023-08-28 16:30"),
    "st6_up":   ("2023-08-28 11:30", "2023-08-28 16:30"),
    "st7_down": ("2023-09-14 13:00", "2023-09-15 13:00"),
    "st7_up":   ("2023-09-14 13:00", "2023-09-15 13:00")
}

# build event dictionary directly (no function)
sonde_events = {}
down = sonde_downstream.sort_index()   
up = sonde_upstream.sort_index()      

for storm_name, (start, end) in storm_windows.items():
    start = pd.Timestamp(start)
    end = pd.Timestamp(end)

    if "down" in storm_name.lower():
        src = down
    elif "up" in storm_name.lower():
        src = up
    else:
        continue
    sonde_events[storm_name] = src.loc[(src.index >= start) & (src.index <= end)].copy()

Calculate HI for sonde data

In [110]:
sonde_results = []

for storm_name, storm_df in sonde_events.items():
    for constituent in ["Turbidity FNU", "fDOM RFU"]:
        if constituent not in storm_df.columns:
            continue

        result = calculate_aich_HI(
            storm_df,
            tau_col="shear_stress",
            constituent_col=constituent,
            storm_name=storm_name)

        if result is not None:
            result["storm"] = storm_name
            result["constituent"] = constituent
            sonde_results.append(result)

sonde_results = pd.DataFrame(sonde_results)
sonde_results.to_csv('HI_calculations/aich_sonde_hysteresis_results.csv', index=False)

Not enough paired data to calculate AICH HI for storm st2_up
Not enough paired data to calculate AICH HI for storm st2_up
Not enough paired data to calculate AICH HI for storm st3_up
Not enough paired data to calculate AICH HI for storm st3_up
Not enough paired data to calculate AICH HI for storm st5_up
Not enough paired data to calculate AICH HI for storm st5_up
Not enough paired data to calculate AICH HI for storm st6_up
Not enough paired data to calculate AICH HI for storm st6_up
Not enough paired data to calculate AICH HI for storm st7_down
Not enough paired data to calculate AICH HI for storm st7_down


Plot sonde data

In [111]:
for storm_name, storm_df in sonde_events.items():
    for constituent in ["Turbidity FNU", "fDOM RFU"]:
        if constituent not in storm_df.columns:
            continue

        result = calculate_aich_HI(
            storm_df,
            tau_col="shear_stress",
            constituent_col=constituent,
            storm_name=storm_name
        )

        if result is not None:
            plot_aich_hysteresis(result, storm_df, "shear_stress", constituent, out_dir="HI_calculations/aich_plots/sonde")

Not enough paired data to calculate AICH HI for storm st2_up
Not enough paired data to calculate AICH HI for storm st2_up
Not enough paired data to calculate AICH HI for storm st3_up
Not enough paired data to calculate AICH HI for storm st3_up
Not enough paired data to calculate AICH HI for storm st5_up
Not enough paired data to calculate AICH HI for storm st5_up
Not enough paired data to calculate AICH HI for storm st6_up
Not enough paired data to calculate AICH HI for storm st6_up
Not enough paired data to calculate AICH HI for storm st7_down
Not enough paired data to calculate AICH HI for storm st7_down
